## Pre-requisites
Install first the libraries needed. This would ideally be done via cluster editing but I don't have those privileges in the free version.

In [0]:
%pip install pystac-client shapely rasterio
dbutils.library.restartPython()

## Bronze layer
Nothing in this layer is "cleaned" yet: the goal of the Bronze layer is to have a reliable, replayable copy of what the source system gave us, so if something breaks downstream, we can reprocess from Bronze instead of re-querying the external API again. 

**Discovery**\
The following STAC scripts are for discovery and checks. I will start a small AOI, a patch of the Bavarian Alps, and I will only take data from June, 2024.

In [0]:
# To check the available collections in the Copernicus STAC API
from pystac_client import Client

catalog = Client.open("https://stac.dataspace.copernicus.eu/v1")

for collection in catalog.get_collections():
    if "sentinel" in collection.id.lower():
        print(f"Collection ID: {collection.id} | Title: {collection.title}")

In [0]:
"""
================================================================================
Bronze Layer Ingestion — Sentinel-2 STAC Metadata
================================================================================
This script takes the metadata (no actual images) of a given region and time 
range, and saves it as a structured table in a delta lake.

"""

import json
from datetime import datetime
from pystac_client import Client
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, ArrayType, TimestampType, MapType
)

# init Spark
spark = SparkSession.builder.getOrCreate()

# -------------------------------------------------------------------------
# 1. PARAMETERS & INPUTS
# -------------------------------------------------------------------------
CATALOG_URL = "https://stac.dataspace.copernicus.eu/v1"
COLLECTION = "sentinel-2-l2a" # Collection ID: sentinel-2-l2a | Title: Sentinel-2 Level-2A
TARGET_DELTA_TABLE = "fca_copernicus.bronze.sentinel2_stac_raw"

# Define Area of Interest: [min_lon, min_lat, max_lon, max_lat]
# Roughly around Tucuman
BBOX = [-66.0, -27.5, -65, -26.5]  # This can be a polygon too

# Define Temporal Window
START_DATE = "2024-06-01"
END_DATE = "2024-06-05"
DATETIME_RANGE = f"{START_DATE}T00:00:00Z/{END_DATE}T23:59:59Z"

# -------------------------------------------------------------------------
# 2. QUERY COPERNICUS STAC API
# -------------------------------------------------------------------------
print(f">>> [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]: BEGIN DATA INGESTION")
print(f"Connecting to Copernicus STAC API at {CATALOG_URL}...")
catalog = Client.open(CATALOG_URL)

search = catalog.search(
    collections=[COLLECTION],
    bbox=BBOX,
    datetime=DATETIME_RANGE,
    limit=100
)

stac_items = list(search.items())
print(f"Retrieved {len(stac_items)} STAC items from CDSE.")

if not stac_items:
    raise ValueError("No STAC items found matching the given criteria.")

# Convert PySTAC Item objects into raw PyDicts
raw_items_dict = [item.to_dict() for item in stac_items]

# -------------------------------------------------------------------------
# 3. SPARK SCHEMA DEFINITION
# -------------------------------------------------------------------------
# Defining explicit schema (for performance and safe dynamic metadata handling)
asset_schema = StructType([
    StructField("href", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("roles", ArrayType(StringType()), True)
])

stac_schema = StructType([
    StructField("id", StringType(), False),
    StructField("type", StringType(), True),
    StructField("stac_version", StringType(), True),
    StructField("collection", StringType(), True),
    StructField("bbox", ArrayType(DoubleType()), True),
    StructField("geometry", StringType(), True),  # Converted to GeoJSON string
    StructField("properties", MapType(StringType(), StringType()), True),
    StructField("assets", MapType(StringType(), asset_schema), True)
])

# Serialize dynamic dict elements (geometry & properties) so Spark can parse reliably
processed_records = []
for d in raw_items_dict:
    record = {
        "id": d.get("id"),
        "type": d.get("type"),
        "stac_version": d.get("stac_version"),
        "collection": d.get("collection"),
        "bbox": [float(b) for b in d.get("bbox", [])],
        "geometry": json.dumps(d.get("geometry")),
        "properties": {k: str(v) for k, v in d.get("properties", {}).items()},
        "assets": {
            k: {
                "href": v.get("href"),
                "type": v.get("type"),
                "title": v.get("title"),
                "roles": v.get("roles")
            } for k, v in d.get("assets", {}).items()
        }
    }
    processed_records.append(record)

# -------------------------------------------------------------------------
# 4. CREATE SPARK DATAFRAME & TRANSFORM TO BRONZE METADATA
# -------------------------------------------------------------------------
df_raw = spark.createDataFrame(processed_records, schema=stac_schema)

df_bronze = (
    df_raw
    # Parse explicit timestamps from STAC properties
    .withColumn("acquisition_time", F.to_timestamp(F.col("properties")["datetime"]))
    # Extract cloud cover for easy spatial filtering downstream
    .withColumn("cloud_cover", F.col("properties")["eo:cloud_cover"].cast("double"))
    # Ingestion Audit Metadata
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_catalog", F.lit(CATALOG_URL))
)
# -------------------------------------------------------------------------
# 5. WRITE TO BRONZE DELTA TABLE IN UNITY CATALOG (FULL OVERWRITE)
# -------------------------------------------------------------------------
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")  # change to "append" when in production
    .option("overwriteSchema", "true")  # allows overwriting table definition if schema changes
    .partitionBy("collection")
    .saveAsTable(TARGET_DELTA_TABLE)
)

print(f"Successfully overwritten {TARGET_DELTA_TABLE} with {df_bronze.count()} records.")
print(f">>> [{datetime.now()}]: DATA INGESTION COMPLETED!")

**Checks:** \
Find, select, and display key spectral bands for downstream Silver processing.

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("fca_copernicus.bronze.sentinel2_stac_raw")
df_bronze.select("id", F.map_keys("assets").alias("available_asset_keys")).show(3, truncate=False)

We can see that of the selected bands for this study, the following are available:

B04_10m, B04_20m, B04_60m\
B08_10m\
B11_20m, B11_60m\
B12_20m, B12_60m\
SCL_20m, SCL_60m\
Check they loaded correctly:

In [0]:

df_bronze = spark.table("fca_copernicus.bronze.sentinel2_stac_raw")

df_bands = df_bronze.select(
    "id",
    "cloud_cover",
    df_bronze.assets["B04_10m"]["href"].alias("red_band_href"),
    df_bronze.assets["B08_10m"]["href"].alias("nir_band_href"),
    df_bronze.assets["B11_20m"]["href"].alias("swir1_band_href"),
    df_bronze.assets["B12_20m"]["href"].alias("swir2_band_href"),
    df_bronze.assets["SCL_20m"]["href"].alias("scl_mask_href")
)

display(df_bands)
# display(df_bronze.limit(10))

**Full data download**\
To the full .SAFE.zip files download, as requested by client.

Stream download naming convention:\
`bronze/raw/sentinel2/{tile_id}/{año_mes_día}/{nombre_producto}/
{nombre_producto}.zip
metadata.json `

This translates to this format in the UC Volume (example):\
`/Volumes/fca_copernicus/bronze/sentinel2_raw/T32TPT/2024_06_04/S2B_MSIL2A_20240604T101559_N0510_R065_T32TPT_20240604T130328/S2B_MSIL2A_20240604T101559_N0510_R065_T32TPT_20240604T130328.zip` 


In [0]:

#========================================================
#========================================================
#========================================================
CDSE_USERNAME = "tatimr93@gmail.com"
CDSE_PASSWORD = "Extortion_Slang_Reboot6_Unvisited"

In [0]:
# ===============================================================================
# CELDA DE TESTING: borra UC Volume + tabla de control antes de una corrida chica
# NO CORRER en producción. Requiere confirmación explícita abajo.
# ===============================================================================

TESTING_MODE = True  # <-- cambiar a False (o borrar la celda) antes de correr en serio

# Confirmación manual: evita que alguien corra "Run All" sin querer y borre todo.
CONFIRM_WIPE = "SI_BORRAR"  # <-- tenés que escribir esto a mano cada vez

if not TESTING_MODE:
    raise RuntimeError("TESTING_MODE está en False — esta celda no debería correr.")

if CONFIRM_WIPE != "SI_BORRAR":
    raise RuntimeError("Confirmación no provista. Editá CONFIRM_WIPE para confirmar el borrado.")

# --- 1. Vaciar el Volume ---
try:
    contents = dbutils.fs.ls(VOLUME_ROOT)
    for item in contents:
        dbutils.fs.rm(item.path, recurse=True)
    print(f"Volume vaciado: {VOLUME_ROOT} ({len(contents)} items borrados)")
except Exception as e:
    print(f"Volume ya estaba vacío o no existe todavía: {e}")

# --- 2. Vaciar la tabla de control (si no, el anti-join va a saltear todo) ---
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

try:
    spark.sql(f"TRUNCATE TABLE {CONTROL_TABLE}")
    print(f"Tabla de control vaciada: {CONTROL_TABLE}")
except Exception as e:
    print(f"Tabla de control no existe todavía (nada que truncar): {e}")

print(">>> Listo para correr una ingesta de prueba desde cero.")

In [0]:
"""
================================================================================
Download of complete .SAFE files (Sentinel-2)
================================================================================
Grabs the indexed STAC API from above and
    (1) resolves the OData Product ID,
    (2) downloads the .SAFE.zip,
    (3) uploads to ADLS Gen2, and
    (4) generate an enriched metadata.json.

I don't have an ADLS account, so I will temporarily use a Unity Catalog Volume.
The resulting structure will be:
      /Volumes/fca_copernicus/bronze/sentinel2_raw/
          {tile_id}/{year_month_day}/{product_name}/
              {product_name}.zip
              metadata.json

This is a temporal patch, once we get an ADLS Gen2 account, this will be
replaced and the function changes to "upload".

--> Ingestion control table:
    Every run reads fca_copernicus.control.ingestion_log to skip scenes
    already ingested successfully (anti-join), and appends new records
    (batched every LOG_FLUSH_EVERY scenes + a final flush) with status:
        - success
        - failed_retryable   -> transient network/HTTP error, worth retrying
        - failed_permanent   -> product not found / bad response, retrying
                                 won't help without upstream changes

"""

import os
import json
import time
import hashlib
import requests
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# -------------------------------------------------------------------------
# 1. Parameters
# -------------------------------------------------------------------------

print(f">>> [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]: BEGIN DATA INGESTION")
print(f"Connecting to CDSE...")

# --- Copernicus authentication ---
CDSE_USERNAME = dbutils.secrets.get(scope="cdse", key="username")
CDSE_PASSWORD = dbutils.secrets.get(scope="cdse", key="password")

CDSE_TOKEN_URL = (
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/"
    "protocol/openid-connect/token"
)
CDSE_ODATA_BASE = "https://catalogue.dataspace.copernicus.eu/odata/v1"

# --- Storage: Unity Catalog Volume ---
UC_CATALOG = "fca_copernicus"
UC_SCHEMA = "bronze"
UC_VOLUME = "sentinel2_raw"
VOLUME_ROOT = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"

# --- Source: indexed Delta table containing all scenes (above) ---
SOURCE_DELTA_TABLE = "fca_copernicus.bronze.sentinel2_stac_raw"

# --- Ingestion control table ---
UC_CONTROL_SCHEMA = "control"
CONTROL_TABLE = f"{UC_CATALOG}.{UC_CONTROL_SCHEMA}.ingestion_log"
LOG_FLUSH_EVERY = 20  # write to the control table every N scenes (plus a final flush)

# --- Retry policy for transient failures (network / 5xx) ---
MAX_RETRIES = 3
RETRY_BACKOFF_BASE_SECONDS = 5  # 5s, 10s, 20s


# -------------------------------------------------------------------------
# 2. OAuth2 authentication
# -------------------------------------------------------------------------
def get_cdse_access_token() -> str:
    resp = requests.post(
        CDSE_TOKEN_URL,
        data={
            "client_id": "cdse-public",
            "username": CDSE_USERNAME,
            "password": CDSE_PASSWORD,
            "grant_type": "password",
        },
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


# -------------------------------------------------------------------------
# 3. OData Product ID
# -------------------------------------------------------------------------
class PermanentIngestionError(Exception):
    """Raised for failures that a retry won't fix (e.g. product doesn't exist,
    checksum/size mismatch after download). These are logged as
    'failed_permanent' and NOT retried within this run."""
    pass

class TransientIngestionError(Exception):
    """Raised for failures worth retrying (network errors, 5xx responses).
    These are logged as 'failed_retryable' if retries are exhausted."""
    pass


def resolve_odata_product_id(product_name: str, token: str) -> dict:
    # Ensure the product name ends with .SAFE for the OData query
    search_name = product_name if product_name.endswith(".SAFE") else f"{product_name}.SAFE"

    filter_query = f"Name eq '{search_name}'"
    url = f"{CDSE_ODATA_BASE}/Products?$filter={filter_query}"

    try:
        resp = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=60)
    except requests.exceptions.RequestException as e:
        raise TransientIngestionError(f"OData query network error: {e}") from e

    if resp.status_code >= 500:
        raise TransientIngestionError(f"OData query returned {resp.status_code}")
    resp.raise_for_status()  # 4xx -> real HTTPError, treated as permanent below

    results = resp.json().get("value", [])

    if not results:
        raise PermanentIngestionError(f"Product not found in OData: {search_name}")

    return results[0]


# -------------------------------------------------------------------------
# 4. tile_id (MGRS) extraction
# -------------------------------------------------------------------------
def extract_mgrs_tile(product_name: str) -> str:
    clean_name = product_name.replace(".SAFE", "")
    parts = clean_name.split("_")
    tile_part = next((p for p in parts if p.startswith("T") and len(p) == 6), None)

    if tile_part is None:
        raise PermanentIngestionError(f"Unable to extract tile_id from: {product_name}")
    return tile_part


# -------------------------------------------------------------------------
# 5. .zip download to Volume (streaming) + checksum
# -------------------------------------------------------------------------
def _extract_expected_md5(odata_product: dict) -> str | None:
    """CDSE OData 'Checksum' is a list like [{'Algorithm': 'MD5', 'Value': '...'}]."""
    for entry in odata_product.get("Checksum", []) or []:
        if entry.get("Algorithm", "").upper() == "MD5":
            return entry.get("Value", "").lower()
    return None


def download_safe_zip(odata_product_id: str, token: str, dest_path: str) -> tuple[int, str]:
    """
    The .SAFE files are quite heavy (up to 1.2 GB ea), so we stream-download
    (chunks of 8 MB). Computes MD5 while streaming so we don't need a second
    pass over the file. 
    Returns (bytes_written, md5_hexdigest).
    
    ===> Change 'dest_path' once we have an ADLS Gen2 account.
    """

    url = f"{CDSE_ODATA_BASE}/Products({odata_product_id})/$value"
    headers = {"Authorization": f"Bearer {token}"}

    session = requests.Session()
    session.headers.update(headers)

    try:
        # Initial request: get redirect location manually if redirected
        response = session.get(url, stream=True, allow_redirects=False, timeout=60)

        if response.status_code in (301, 302, 303, 307):
            redirect_url = response.headers["Location"]
            # Make second request to actual storage node retaining Bearer token
            response = session.get(redirect_url, stream=True, timeout=60)

        if response.status_code >= 500:
            raise TransientIngestionError(f"Download returned {response.status_code}")
        response.raise_for_status()

        bytes_written = 0
        md5_hash = hashlib.md5()
        with open(dest_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:  # filter out keep-alive chunks
                    f.write(chunk)
                    md5_hash.update(chunk)
                    bytes_written += len(chunk)

        return bytes_written, md5_hash.hexdigest()

    except requests.exceptions.RequestException as e:
        # Network drop mid-stream, connection reset, timeout, etc. -> transient
        _cleanup_partial_file(dest_path)
        raise TransientIngestionError(f"Download network error: {e}") from e


def _cleanup_partial_file(path: str):
    """Remove a partially-written / corrupt zip so it doesn't linger in the
    Volume and doesn't get mistaken for a complete file on a future run."""
    try:
        if os.path.exists(path):
            os.remove(path)
            print(f"  -> cleaned up partial file: {path}")
    except OSError as e:
        print(f"  -> WARNING: could not remove partial file {path}: {e}")


def verify_download(bytes_written: int, actual_md5: str, odata_product: dict, dest_path: str):
    """Raises PermanentIngestionError (and deletes the bad file) if size or
    checksum don't match what OData reported for the product."""
    expected_size = odata_product.get("ContentLength")
    if expected_size is not None and bytes_written != expected_size:
        _cleanup_partial_file(dest_path)
        raise PermanentIngestionError(
            f"Size mismatch: expected {expected_size} bytes, got {bytes_written}"
        )

    expected_md5 = _extract_expected_md5(odata_product)
    if expected_md5 is not None and actual_md5 != expected_md5:
        _cleanup_partial_file(dest_path)
        raise PermanentIngestionError(
            f"Checksum mismatch: expected MD5 {expected_md5}, got {actual_md5}"
        )


# -------------------------------------------------------------------------
# 6. Build enriched metadata.json
# -------------------------------------------------------------------------
def build_enriched_metadata(stac_row: dict, odata_product: dict, bytes_written: int, actual_md5: str) -> dict:
    return {
        "product_name": stac_row["id"],
        "collection": stac_row["collection"],
        "tile_id": extract_mgrs_tile(stac_row["id"]),
        "acquisition_time": stac_row["acquisition_time"],
        "cloud_cover_pct": stac_row["cloud_cover"],
        "bbox": stac_row["bbox"],
        "odata_product_id": odata_product["Id"],
        "content_length_bytes": odata_product.get("ContentLength"),
        "checksum": odata_product.get("Checksum"),
        "downloaded_bytes": bytes_written,
        "downloaded_md5": actual_md5,
        "source_catalog_stac": "https://stac.dataspace.copernicus.eu/v1",
        "source_catalog_odata": CDSE_ODATA_BASE,
        "storage_backend": "unity_catalog_volume",  # key for traceability
        "ingested_at": datetime.utcnow().isoformat() + "Z",
    }


# -------------------------------------------------------------------------
# 7. Ingestion control table helpers
# -------------------------------------------------------------------------
def ensure_control_schema_exists(spark):
    """Creates the control schema if it doesn't exist."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_CONTROL_SCHEMA}")


def get_already_ingested(spark):
    """Anti-join source: scenes tagged as status = 'success' are excluded."""
    if not spark.catalog.tableExists(CONTROL_TABLE):
        return spark.createDataFrame([], "product_name string")

    return (
        spark.table(CONTROL_TABLE)
        .filter("status = 'success'")
        .select("product_name")
        .distinct()
    )


def flush_log_records(spark, log_records: list):
    """Batched append to the control table. Called every LOG_FLUSH_EVERY
    scenes and once more at the end, instead of one write per scene, to
    avoid the small-file problem on Delta."""
    if not log_records:
        return
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    
    schema = StructType([
        StructField("product_name", StringType(), True),
        StructField("checksum", StringType(), True),
        StructField("tile_id", StringType(), True),
        StructField("acquisition_date", StringType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("file_size_bytes", LongType(), True),
        StructField("storage_path", StringType(), True),
        StructField("ingestion_started_at", TimestampType(), True),
        StructField("ingestion_completed_at", TimestampType(), True),
        StructField("databricks_run_id", StringType(), True),
    ])
    
    # Convert checksum to JSON string for storage
    import json
    for record in log_records:
        if record.get("checksum") is not None:
            record["checksum"] = json.dumps(record["checksum"])
    
    log_df = spark.createDataFrame(log_records, schema=schema)
    (log_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(CONTROL_TABLE))
    print(f"  [log] flushed {len(log_records)} record(s) to {CONTROL_TABLE}")
    log_records.clear()

def get_run_id(spark) -> str:
    """Added to deal with spark.conf.get errors (default failed)"""
    try:
        return spark.conf.get("spark.databricks.job.runId")
    except Exception:
        return "interactive"

def make_log_record(product_name, checksum, tile_id, acquisition_date, status,
                     error_message, file_size_bytes, storage_path,
                     started_at, run_id):
    return {
        "product_name": product_name,
        "checksum": checksum,
        "tile_id": tile_id,
        "acquisition_date": acquisition_date,
        "status": status, 
        "error_message": error_message,
        "file_size_bytes": file_size_bytes,
        "storage_path": storage_path,
        "ingestion_started_at": started_at,
        "ingestion_completed_at": datetime.utcnow(),
        "databricks_run_id": run_id,
    }


# -------------------------------------------------------------------------
# 8. Orchestration
# -------------------------------------------------------------------------
def main():
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    run_id = get_run_id(spark)

    ensure_control_schema_exists(spark)  

    already_ingested = get_already_ingested(spark)

    source_df = spark.table(SOURCE_DELTA_TABLE)
    scenes_df = source_df.join(already_ingested, source_df["id"] == already_ingested["product_name"], how="left_anti")  # skip already-successful scenes
    scenes = [row.asDict() for row in scenes_df.collect()]
    print(f"{len(scenes)} scenes pending ingestion")

    token = get_cdse_access_token()
    token_issued_at = time.time()
    log_records = []

    for i, scene in enumerate(scenes):
        if time.time() - token_issued_at > 540:  # CDSE tokens expire at 10 min; refresh at 9 min
            token = get_cdse_access_token()
            token_issued_at = time.time()

        raw_id = scene["id"]
        product_name = raw_id.replace(".SAFE", "")
        started_at = datetime.utcnow()
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]: [{i+1}/{len(scenes)}] Processing {product_name}...")

        tile_id = None
        acquisition_date = None
        product_dir = None
        attempt = 0

        while True:
            attempt += 1
            try:
                odata_product = resolve_odata_product_id(product_name, token)
                tile_id = extract_mgrs_tile(product_name)

                acq_time = scene["acquisition_time"]
                if isinstance(acq_time, str):
                    acq_time = datetime.fromisoformat(acq_time.replace("Z", "+00:00"))
                acquisition_date = acq_time.strftime("%Y_%m_%d")

                product_dir = f"{VOLUME_ROOT}/{tile_id}/{acquisition_date}/{product_name}"
                os.makedirs(product_dir, exist_ok=True)
                zip_dest = f"{product_dir}/{product_name}.zip"
                meta_dest = f"{product_dir}/metadata.json"

                bytes_written, actual_md5 = download_safe_zip(odata_product["Id"], token, zip_dest)
                verify_download(bytes_written, actual_md5, odata_product, zip_dest)

                metadata = build_enriched_metadata(scene, odata_product, bytes_written, actual_md5)
                with open(meta_dest, "w") as f:
                    json.dump(metadata, f, indent=2, default=str)

                print(f"  -> OK: written in {product_dir}/ ({bytes_written*1e-9} GB, md5 verified)")

                log_records.append(make_log_record(
                    product_name, odata_product.get("Checksum"), tile_id, acquisition_date,
                    "success", None, bytes_written, product_dir, started_at, run_id
                ))
                break  # success, exit retry loop

            except PermanentIngestionError as e:
                print(f"  -> PERMANENT FAILURE in {product_name}: {e}")
                log_records.append(make_log_record(
                    product_name, None, tile_id, acquisition_date,
                    "failed_permanent", str(e)[:500], None, product_dir, started_at, run_id
                ))
                break  # don't retry

            except TransientIngestionError as e:
                if attempt >= MAX_RETRIES:
                    print(f"  -> TRANSIENT FAILURE in {product_name} (gave up after {attempt} attempts): {e}")
                    log_records.append(make_log_record(
                        product_name, None, tile_id, acquisition_date,
                        "failed_retryable", str(e)[:500], None, product_dir, started_at, run_id
                    ))
                    break
                backoff = RETRY_BACKOFF_BASE_SECONDS * (2 ** (attempt - 1))
                print(f"  -> transient error (attempt {attempt}/{MAX_RETRIES}), retrying in {backoff}s: {e}")
                time.sleep(backoff)
                continue  # retry

            except Exception as e:
                # Anything unexpected: treat as retryable-exhausted, don't loop forever on unknowns
                print(f"  -> UNEXPECTED ERROR in {product_name}: {e}")
                log_records.append(make_log_record(
                    product_name, None, tile_id, acquisition_date,
                    "failed_permanent", f"unexpected: {str(e)[:480]}", None, product_dir, started_at, run_id
                ))
                break

        if len(log_records) >= LOG_FLUSH_EVERY:
            flush_log_records(spark, log_records)

    flush_log_records(spark, log_records)  # final flush for the remainder


if __name__ == "__main__":
    main()
    print(f">>> [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]: END DATA INGESTION")